In [1]:
from pyspark.sql import SparkSession
import pyspark
print(pyspark.__version__)
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F
from urllib.parse import urlparse
from datetime import date
from utils import find_new_paths

3.4.1


In [2]:
spark = (SparkSession.builder
    .appName("preprocess_batch")
    .master("spark://spark-master:7077")
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/nifi/crypto-prices")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.hadoop.hive.exec.dynamic.partition", "true")
    .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict")
    .getOrCreate())

spark.sql("USE DATABASE CryptoPredictions")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/17 19:26:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/17 19:26:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/17 19:26:53 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/17 19:27:03 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

# Preprocessing całego folderu i zapis do Hive

In [3]:
BASE_PATH = "hdfs://namenode:8020/nifi/crypto-prices/"
META_PATH = "hdfs://namenode:8020/nifi/metadata/crypto_prices_last_path.txt"

paths_to_read, new_dirs = find_new_paths(spark, BASE_PATH, META_PATH)

Last path: 2025-12-15
Removing 2026-01-17 from dirs - day has not finished yet


In [8]:
def rename_and_transform(df):
    # Wyciągnięcie symbolu
    df = df.withColumn(
        "Symbol",
        F.split("symbol", ":")[1].substr(1, 3)
    )

    # Timestamp → Datetime
    df = df.withColumn(
        "Datetime",
        F.from_unixtime(F.col("timestamp")).cast("timestamp")
    )
    
    # Mapowanie: stara_nazwa → nowa_nazwa
    rename_map = {
        "current_price": "CurrentPrice",
        "open": "OpeningPrice",
        "low": "LowestDayPrice",
        "high": "HighestDayPrice",
        "previous_close": "PreviousClosingPrice"
    }

    # Zmiana nazw
    df = df.withColumnsRenamed(rename_map)
    
    df = df.withColumn("PartitionDate", F.to_date("Datetime"))
    df= df.filter(df.PartitionDate.isNotNull())
    
    return df.select(
        "Symbol",
        "CurrentPrice",
        "OpeningPrice",
        "LowestDayPrice",
        "HighestDayPrice",
        "PreviousClosingPrice",
        "Datetime",
        "PartitionDate"
    )

df = spark.read.parquet(*paths_to_read)
df = rename_and_transform(df)

# print(df.count())
df.describe().show()
df.sort(df.Datetime.desc()).show(5)

+-------+------+------------------+------------------+------------------+------------------+--------------------+
|summary|Symbol|      CurrentPrice|      OpeningPrice|    LowestDayPrice|   HighestDayPrice|PreviousClosingPrice|
+-------+------+------------------+------------------+------------------+------------------+--------------------+
|  count|547679|            547679|            547679|            547679|            547679|              547679|
|   mean|  null|31016.786811508184|30943.372718143335|30574.281814785645|31450.562171454894|  30943.372521714413|
| stddev|  null| 41659.05637051154|41557.270752447854| 41072.18579786904| 42228.89300522943|   41557.27048608522|
|    min|   BTC|            117.08|            116.97|            116.88|            123.46|              116.97|
|    max|   SOL|          97899.48|          97865.76|           95777.0|          97924.49|            97865.75|
+-------+------+------------------+------------------+------------------+---------------

[Stage 11:======================================================> (28 + 1) / 29]

+------+------------+------------+--------------+---------------+--------------------+-------------------+-------------+
|Symbol|CurrentPrice|OpeningPrice|LowestDayPrice|HighestDayPrice|PreviousClosingPrice|           Datetime|PartitionDate|
+------+------------+------------+--------------+---------------+--------------------+-------------------+-------------+
|   ETH|     3295.52|     3314.32|       3253.01|        3326.79|             3314.32|2026-01-16 23:19:07|   2026-01-16|
|   SOL|      144.98|      142.56|        140.26|         145.55|              142.55|2026-01-16 23:19:07|   2026-01-16|
|   BTC|    95496.01|    95686.72|      94293.46|       95871.47|            95686.72|2026-01-16 23:19:07|   2026-01-16|
|   SOL|      145.05|      142.55|        140.26|         145.55|              142.56|2026-01-16 23:18:48|   2026-01-16|
|   ETH|     3297.04|     3314.33|       3253.01|        3326.79|             3314.32|2026-01-16 23:18:48|   2026-01-16|
+------+------------+-----------

In [9]:
# Sanity Check
bounds = (
    df.select(
        F.min("Datetime").alias("first_datetime"),
        F.max("Datetime").alias("last_datetime")
    )
    .collect()[0]
)

print(f"[SANITY CHECK] Datetime range: {bounds.first_datetime} → {bounds.last_datetime}")

[Stage 12:======================================================> (28 + 1) / 29]

[SANITY CHECK] Datetime range: 2025-12-15 23:15:48 → 2026-01-16 23:19:07


In [7]:
(df.write
  .mode("append")
  .format("hive")
  .partitionBy("PartitionDate")
  .saveAsTable("CryptocurrencySnapshot"))

latest_dir = max(new_dirs)

spark.createDataFrame(
    [(latest_dir,)],
    ["last_processed_path"]
).write.mode("overwrite").text(META_PATH)

print(f"Created last path file: {latest_dir}")

Created last path file: 2026-01-16


In [13]:
spark.sql("""SELECT * FROM CryptocurrencySnapshot
            ORDER BY PartitionDate DESC
            LIMIT 10""").show()

[Stage 17:======================================================> (32 + 1) / 33]

+------+-------------------+------------+------------+--------------+---------------+--------------------+-------------+
|Symbol|           Datetime|CurrentPrice|OpeningPrice|LowestDayPrice|HighestDayPrice|PreviousClosingPrice|PartitionDate|
+------+-------------------+------------+------------+--------------+---------------+--------------------+-------------+
|   ETH|2026-01-16 21:39:25|     3296.92|     3298.79|       3253.01|        3326.79|              3298.8|   2026-01-16|
|   SOL|2026-01-16 22:29:19|      144.89|      142.45|        140.26|         145.55|              142.45|   2026-01-16|
|   BTC|2026-01-16 02:28:12|    95564.97|    96535.23|      95134.48|       97193.34|            96535.24|   2026-01-16|
|   ETH|2026-01-16 22:29:58|     3288.27|     3308.27|       3253.01|        3326.79|             3308.26|   2026-01-16|
|   BTC|2026-01-16 03:18:11|    95527.73|    96290.93|      95134.48|       97193.34|            96290.93|   2026-01-16|
|   ETH|2026-01-16 22:29:19|    

In [8]:
# Do testów
META_PATH = "hdfs://namenode:8020/nifi/metadata/crypto_prices_last_path.txt"
last_path = spark.read.text(META_PATH).first()[0]
print(last_path)

2026-01-16


In [9]:
spark.stop()